# 01 – Rohdaten explorieren

Phase 3 des Leitfadens: Daten verstehen bevor Features gebaut werden.  
Ziel: Lücken, Ausreißer und Formatprobleme visuell erkennen.

In [ ]:
import sys
from pathlib import Path

# Projektpfade bekannt machen
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from parse_health_xml import extract_records, filter_records
from build_daily_metrics import build_daily_metrics
from config import XML_FILE

sns.set_theme(style='whitegrid')
print('Setup OK – XML-Datei:', XML_FILE)

## 1 – XML laden & erste Inspektion

In [ ]:
records = extract_records()
print(f'Gesamte Records: {len(records)}')

# Welche Typen gibt es überhaupt?
import collections
type_counts = collections.Counter(r['type'] for r in records)
for t, n in sorted(type_counts.items(), key=lambda x: -x[1])[:20]:
    print(f'  {n:>8}  {t}')

In [ ]:
# Erste 3 Records anzeigen
records[:3]

## 2 – Tagesmetriken aufbauen (Step 12)

In [ ]:
df = build_daily_metrics(records)
df['date'] = pd.to_datetime(df['date'])
df.head(10)

In [ ]:
print(df.shape)
print('\nDatentypen:')
print(df.dtypes)
print('\nFehlende Werte:')
print(df.isna().sum())
print('\nStatistik:')
df.describe()

## 3 – Zeitreihen visualisieren (Step 11)

Achte auf: Lücken im Datensatz, Ausreißer, plausible Wertebereiche.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Ruhepuls
axes[0].plot(df['date'], df['resting_hr'], color='tomato', linewidth=1)
axes[0].set_ylabel('Ruhepuls (bpm)')
axes[0].set_title('Ruhepuls über Zeit')

# HRV
axes[1].plot(df['date'], df['hrv'], color='steelblue', linewidth=1)
axes[1].set_ylabel('HRV (ms)')
axes[1].set_title('HRV über Zeit')

# Schlafstunden
axes[2].bar(df['date'], df['sleep_hours'], color='mediumpurple', width=0.8)
axes[2].axhline(7, color='gray', linestyle='--', linewidth=0.8, label='7 h Ziel')
axes[2].set_ylabel('Schlaf (h)')
axes[2].set_title('Schlafstunden über Zeit')
axes[2].legend()

# Workout-Minuten
axes[3].bar(df['date'], df['workout_minutes'], color='seagreen', width=0.8)
axes[3].set_ylabel('Workout (min)')
axes[3].set_title('Workout-Minuten über Zeit')
axes[3].set_xlabel('Datum')

plt.tight_layout()
plt.show()

## 4 – Korrelationsmatrix

In [ ]:
numeric_cols = ['resting_hr', 'hrv', 'sleep_hours', 'workout_minutes']
corr = df[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Korrelationsmatrix der Tagesmetriken')
plt.tight_layout()
plt.show()

## 5 – Ausreißer & fehlende Zeiträume prüfen

In [ ]:
# Lücken im Datensatz: Tage ohne Eintrag
if len(df) > 1:
    all_days = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
    missing_days = all_days.difference(df['date'])
    print(f'Fehlende Tage im Zeitraum: {len(missing_days)}')
    if len(missing_days) > 0:
        print(missing_days[:10].tolist())

# Plausibilitäts-Check
print('\n--- Ruhepuls ---')
print(df['resting_hr'].describe())
print('Werte < 30 bpm:', (df['resting_hr'] < 30).sum())
print('Werte > 120 bpm:', (df['resting_hr'] > 120).sum())

print('\n--- Schlaf ---')
print(df['sleep_hours'].describe())
print('Werte < 1 h:', (df['sleep_hours'] < 1).sum())
print('Werte > 16 h:', (df['sleep_hours'] > 16).sum())